# Автоматичний датасет best2 з готових відео Google Drive

Ноутбук читає спільну теку Drive, визначає клас за суфіксом відео (`analog`, `digital`, `thermal_analog`, `thermal_digital`) або за назвою теки. Витягує кадри з кроком 0.5 секунди та формує `train/val/test`, не розділяючи парні експорти одного джерела. Якщо у Drive є `best2_thermal_v1.zip`, зберігає його split-и для знайомих джерел. ZIP зберігається окремим файлом у `MyDrive/best2_training/input/`.

Спершу перевір звіт про класи й список нерозпізнаних назв. Відео без однозначної мітки ноутбук пропускає. Це створює класи для класифікатора; автоматичної розмітки логотипів тут немає.


In [ ]:
%pip -q install google-api-python-client opencv-python-headless pillow

from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth import default
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image
import cv2, csv, hashlib, io, math, random, re, shutil, time, zipfile, unicodedata

auth.authenticate_user()
drive.mount('/content/drive')
creds, _ = default()
service = build('drive', 'v3', credentials=creds)

# Посилання на спільну теку з уже підготовленими відео
SOURCE_FOLDER_ID = '1p7uEPZ8WIWWvzbAmy9sV3-L4hWLj-ieb'
OUTPUT_DIR = Path('/content/best2_thermal_v2')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/best2_training/input')
ARCHIVE_PATH = DRIVE_OUTPUT_DIR / 'best2_thermal_v2.zip'
SAMPLE_SECONDS = 0.5  # 2 кадри/секунду
MAX_EDGE = 1024
JPEG_QUALITY = 85
MAX_DATASET_GIB = 1.8
SPLIT_RATIOS = {'train': 0.70, 'val': 0.15, 'test': 0.15}
RANDOM_SEED = 20261030
CLASSES = ('analog', 'digital', 'thermal_analog', 'thermal_digital')
print('Параметри задані; джерело:', SOURCE_FOLDER_ID)


In [ ]:
# Знаходимо відео рекурсивно. Мітки не вгадуємо: тільки назва класової теки
# або суфікс імені файла, наприклад *_crop_thermal_analog.mp4.
FOLDER_MIME = 'application/vnd.google-apps.folder'
VIDEO_EXTS = {'.mp4', '.mov', '.m4v', '.mkv', '.webm', '.mpeg', '.mpg'}
LABEL_RE = re.compile(r'(?:^|[_\-\s])(?:crop[_\-\s]?)?(thermal[_\-\s]?(?:analog|digital)|analog|digital)$', re.I)

def list_children(folder_id):
    token = None
    while True:
        page = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields='nextPageToken,files(id,name,mimeType,size)', pageSize=1000,
            pageToken=token, supportsAllDrives=True, includeItemsFromAllDrives=True
        ).execute()
        yield from page.get('files', [])
        token = page.get('nextPageToken')
        if not token:
            break

def normalize_class(value):
    value = re.sub(r'[_\-\s]+', '_', value.lower())
    return value if value in CLASSES else None

def label_for(name, folders):
    for folder in reversed(folders):
        label = normalize_class(folder)
        if label:
            return label
    match = LABEL_RE.search(Path(name).stem.lower())
    return normalize_class(match.group(1)) if match else None

videos, unknown, seen = [], [], set()
def walk(folder_id, folders=()):
    for item in list_children(folder_id):
        if item['mimeType'] == FOLDER_MIME:
            walk(item['id'], folders + (item['name'],))
        elif item['mimeType'].startswith('video/') or Path(item['name']).suffix.lower() in VIDEO_EXTS:
            if item['id'] in seen:
                continue
            seen.add(item['id'])
            row = {'id': item['id'], 'name': item['name'], 'class': label_for(item['name'], folders),
                   'size': int(item.get('size') or 0), 'folder': '/'.join(folders)}
            (videos if row['class'] else unknown).append(row)

walk(SOURCE_FOLDER_ID)
print('Відео з міткою:', len(videos), dict(Counter(v['class'] for v in videos)))
print('Відео без мітки:', len(unknown))
for v in unknown[:40]:
    print('ПЕРЕВІР НАЗВУ:', v['folder'], v['name'])
missing = set(CLASSES) - {v['class'] for v in videos}
if missing:
    print('Увага, не знайдено відео класів:', sorted(missing))
if not videos:
    raise RuntimeError('Не знайдено відео з розпізнаними мітками. Перевір назви файлів/тек.')


In [ ]:
# Кожне відео тимчасово завантажується в Colab; кадри зберігаються локально,
# щоб не читати тисячі маленьких файлів безпосередньо з Google Drive.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
for split in SPLIT_RATIOS:
    for cls in CLASSES:
        (OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)
CACHE = Path('/content/video_cache')
CACHE.mkdir(exist_ok=True)

def source_group(name):
    stem = Path(name).stem.lower()
    stem = re.sub(r'(?:[_\-\s]+crop)?[_\-\s]+(?:thermal[_\-\s]?(?:analog|digital)|analog|digital)$', '', stem, flags=re.I)
    stem = unicodedata.normalize('NFKC', stem).casefold()
    return re.sub(r'[^\w]+', '_', stem, flags=re.UNICODE).strip('_') or Path(name).stem

# Зберігаємо split джерел із попереднього ZIP, щоб кадри того самого відео
# не перейшли з test у train під час наступного збору датасету.
PREVIOUS_ARCHIVE = DRIVE_OUTPUT_DIR / 'best2_thermal_v1.zip'
previous_splits = {}
if PREVIOUS_ARCHIVE.is_file():
    with zipfile.ZipFile(PREVIOUS_ARCHIVE) as old_zip:
        manifest_name = next((n for n in old_zip.namelist() if n.endswith('manifest.csv')), None)
        if manifest_name:
            rows = csv.DictReader(old_zip.read(manifest_name).decode('utf-8-sig').splitlines())
            for row in rows:
                previous_splits[row.get('source_group', '')] = row.get('split', '')
    print('З попереднього датасету збережено split джерел:', len(previous_splits))
else:
    print('Попередній ZIP не знайдений; split нових джерел буде створено з нуля.')

groups = sorted({source_group(v['name']) for v in videos})
group_split = {g: previous_splits[g] for g in groups if previous_splits.get(g) in SPLIT_RATIOS}
new_groups = [g for g in groups if g not in group_split]
random.seed(RANDOM_SEED)
random.shuffle(new_groups)
for i, group in enumerate(new_groups):
    p = (i + 0.5) / max(len(new_groups), 1)
    group_split[group] = 'train' if p < .70 else ('val' if p < .85 else 'test')
print('Груп джерел:', len(groups), '| старих split:', len(groups)-len(new_groups), '| нових:', len(new_groups))

def download_video(video, target):
    request = service.files().get_media(fileId=video['id'], supportsAllDrives=True)
    with open(target, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request, chunksize=8*1024*1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()

manifest = []
hashes = set()
saved_bytes = 0
target_bytes = int(MAX_DATASET_GIB * 1024**3)
# За ліміту розміру спершу обробляємо термал, бо це пріоритет датасету.
ordered = sorted(videos, key=lambda v: (not v['class'].startswith('thermal_'), v['name'].lower()))
for index, video in enumerate(ordered, 1):
    group = source_group(video['name'])
    split = group_split[group]
    local = CACHE / (re.sub(r'[^A-Za-z0-9._-]+', '_', video['name'])[:120] or f'{index}.mp4')
    try:
        download_video(video, local)
        cap = cv2.VideoCapture(str(local))
        fps, nframes = cap.get(cv2.CAP_PROP_FPS), cap.get(cv2.CAP_PROP_FRAME_COUNT)
        duration = nframes / fps if fps and fps > 0 else 0
        if not cap.isOpened() or duration <= 0:
            print('Пропущено: відео не читається:', video['name'])
            cap.release()
            continue
        for frame_no in range(math.ceil(duration / SAMPLE_SECONDS)):
            seconds = frame_no * SAMPLE_SECONDS
            cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
            ok, frame = cap.read()
            if not ok or frame is None:
                continue
            h, w = frame.shape[:2]
            scale = min(1.0, MAX_EDGE / max(h, w))
            if scale < 1:
                frame = cv2.resize(frame, (round(w*scale), round(h*scale)), interpolation=cv2.INTER_AREA)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            buf = io.BytesIO()
            Image.fromarray(rgb).save(buf, format='JPEG', quality=JPEG_QUALITY, optimize=True)
            payload = buf.getvalue()
            digest = hashlib.sha1(payload).hexdigest()
            if digest in hashes:
                continue
            if saved_bytes + len(payload) > target_bytes:
                print('Досягнуто ліміт MAX_DATASET_GIB; завершую збір.')
                break
            hashes.add(digest)
            out = OUTPUT_DIR / split / video['class'] / f'{group}_{int(seconds*1000):09d}_{digest[:8]}.jpg'
            out.write_bytes(payload)
            saved_bytes += len(payload)
            manifest.append({'image': str(out.relative_to(OUTPUT_DIR)), 'class': video['class'],
                'split': split, 'source_group': group, 'source_video': video['name'],
                'drive_file_id': video['id'], 'timestamp_seconds': round(seconds, 3), 'sha1': digest})
        cap.release()
        if index % 10 == 0:
            print(f'{index}/{len(ordered)} відео; кадрів {len(manifest)}; {saved_bytes/1024**3:.2f} GiB')
    except Exception as error:
        print('Помилка відео:', video['name'], type(error).__name__, str(error)[:250])
    finally:
        local.unlink(missing_ok=True)
    if saved_bytes >= target_bytes:
        break

if not manifest:
    raise RuntimeError('Кадрів не створено. Перевір доступ до Drive та формати відео.')
with open(OUTPUT_DIR/'manifest.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=manifest[0].keys())
    writer.writeheader()
    writer.writerows(manifest)
counts = defaultdict(Counter)
for row in manifest:
    counts[row['split']][row['class']] += 1
print('Кадрів:', len(manifest), '| розмір:', round(saved_bytes/1024**3, 2), 'GiB')
for split in SPLIT_RATIOS:
    print(split, dict(counts[split]))
empty = [f'{s}/{c}' for s in SPLIT_RATIOS for c in CLASSES if counts[s][c] == 0]
if empty:
    print('УВАГА: порожні класи у split:', empty, '— перевір кількість окремих вихідних відео.')


In [ ]:
# Зберігаємо окремий архів у Drive; попередній ZIP не перезаписується.
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH = DRIVE_OUTPUT_DIR / f'best2_thermal_v2_{time.strftime("%Y%m%d_%H%M%S")}.zip'
with zipfile.ZipFile(ARCHIVE_PATH, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
    for file in OUTPUT_DIR.rglob('*'):
        if file.is_file():
            archive.write(file, Path('best2_thermal_v2') / file.relative_to(OUTPUT_DIR))
print('Готовий ZIP:', ARCHIVE_PATH)
print('Відео без автоматичної мітки:', len(unknown), '— звіт вище.')
